<a id="inizio"></a>
# 11 · Capstone: il carico del Nord e la temperatura

<div style="background:rgba(96,125,139,0.08); border-left:4px solid #607d8b; padding:12px 16px; border-radius:4px; margin:8px 0 12px 0;">

**Blocco 4 · Agenti per il coding e caso d'uso end-to-end · Giornata 2 · Aula Base &nbsp;·&nbsp; ⏱ ~110 min**

Due anni di carico elettrico della zona Nord e la temperatura di Milano: come il carico dipende dall'ora, dal giorno della settimana e dalla temperatura.

**In questo notebook impariamo a**

- caricare, unire e pulire una serie temporale reale, con i salti dell'ora legale
- costruire la serie giornaliera e i profili medi per ora e per giorno della settimana
- unire il carico con la temperatura di Open-Meteo e leggere la relazione tra i due

Prima di questo: [10 · Agenti per il coding](10_Agenti_per_il_coding.ipynb) &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/load_total_north_hourly_2025.xlsx`, `../Dati/fallback/meteo_milano_2024_2025.json`

</div>

<a id="indice"></a>
**Indice**

- [1. Caricamento dei dati e unione](#sez-1)
    - [Step 1 · Caricamento dei dati e unione](#es-1)
- [2. Colonna temporale e prima visualizzazione](#sez-2)
    - [Step 2 · Preparazione della colonna temporale e prima visualizzazione](#es-2)
- [3. Ora legale: duplicati e buchi](#sez-3)
    - [Step 3 · Pulizia dei duplicati dell'ora legale](#es-3)
- [4. Aggregazioni temporali e profili medi](#sez-4)
    - [Step 4 · Aggregazioni temporali e profili medi](#es-4)
- [5. Recupero e integrazione della temperatura](#sez-5)
    - [Step 5 · Recupero e integrazione della temperatura](#es-5)
- [6. Carico e temperatura](#sez-6)
    - [Step 6 · Carico e temperatura](#es-6)

I dati vengono dal [Download Center di Terna](https://dati.terna.it/en/download-center): il **carico
elettrico della zona Nord** negli anni 2024 e 2025. Nei dataset Terna il carico è la potenza richiesta
al sistema elettrico in ciascun intervallo temporale, una grandezza centrale per la pianificazione e la
gestione della rete di trasmissione.

Accanto al carico integriamo la **temperatura** di Milano, presa tramite API da
[Open-Meteo](https://open-meteo.com). L'obiettivo è descrivere come il carico del Nord dipende dall'ora
del giorno, dal giorno della settimana e dalla temperatura.

*Copilot è ammesso per i titoli e le etichette dei grafici; il codice degli step lo scriviamo noi.*

<a id="sez-1"></a>
## 1. Caricamento dei dati e unione

<a id="es-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 1 · Caricamento dei dati e unione**

In questo step costruiamo una singola serie storica continua a partire dai dati grezzi.

**Cosa fare.**

1. Leggi i due file Excel presenti nella cartella `../Dati/` in `df_2024` e `df_2025`:
   `load_total_north_hourly_2024.xlsx` e `load_total_north_hourly_2025.xlsx`.
2. Unisci i due dataset **per righe** in `df`, mantenendo lo stesso schema di colonne, in modo da
   ottenere una serie unica che copra l'intero periodo 2024-2025.
3. Controlla dimensione del dataset, prima e ultima data, quante righe ci sono in media per giorno
   e i valori mancanti per colonna. Il file si chiama `hourly`: la frequenza è oraria?

*Suggerimento:* `df["Date"].dt.date.nunique()` conta i giorni diversi.

</div>

In [ ]:
import pandas as pd

# 1. i due file, uno per anno
df_2024 = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
df_2025 = ...

# 2. uno sotto l'altro, con l'indice rifatto da 0
df = pd.concat([...], ignore_index=True)

# 3. dimensione, prima e ultima data, righe per giorno, valori mancanti
print(df.shape)
print(df["Date"].min(), ...)
righe_giorno = len(df) / df["Date"].dt.date.nunique()
print(f"{righe_giorno:.0f} righe al giorno")
df.isna().sum()

In [ ]:
# Verifica: esegui senza modificare
assert df.shape == (70176, 4), "❌ df: 70.176 righe e 4 colonne, i due anni uno sotto l'altro"
assert df.index[-1] == 70175, "❌ df: l'indice va rifatto da 0, con ignore_index=True in pd.concat"
assert round(righe_giorno) == 96, "❌ righe_giorno: righe totali diviso il numero di giorni diversi"
print("✅ Step 1 completato")

96 righe al giorno: i dati sono quartorari, un valore ogni 15 minuti, anche se il nome del file dice
`hourly`.

<a id="sez-2"></a>
## 2. Colonna temporale e prima visualizzazione

<a id="es-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 2 · Preparazione della colonna temporale e prima visualizzazione**

In questo step rendiamo la colonna data utilizzabile come variabile temporale e facciamo una prima ispezione visiva della serie.

**Cosa fare.**

1. Identifica la colonna che contiene la data/ora e assicurati che sia in formato `datetime`: se la
   data è letta come stringa, la convertiamo con `pd.to_datetime(...)`.
2. Ordina il dataset per data (i file partono dall'ultimo quarto d'ora dell'anno) e reimposta
   l'indice per avere un ordinamento pulito.
3. Importa Plotly e visualizza **entrambe** le serie nel tempo: `Total Load [MW]` e
   `Forecast Total Load [MW]`.
4. Dopo aver verificato che le due curve siano coerenti, elimina le colonne che non useremo:
   `Forecast Total Load [MW]` e `Bidding Zone`.

Al termine di questo step deve rimanere una serie storica con una colonna temporale in formato
`datetime` e una sola colonna di valori, `Total Load [MW]`.

*Suggerimento:* `reset_index(drop=True)` rifà l'indice da 0 senza tenere il vecchio come colonna.

</div>

In [ ]:
import plotly.express as px

# 1. la colonna Date in formato datetime
df["Date"] = pd.to_datetime(df["Date"])

# 2. ordine per data e indice da 0
df = df.sort_values(...).reset_index(drop=True)

# 3. le due serie nel tempo
fig = px.line(df, x="Date", y=[..., ...], title="Carico e previsione di Terna, zona Nord",
              labels={"value": "carico [MW]", "variable": "serie"})
fig.show()

# 4. via le colonne che non useremo
df = df.drop(columns=[...])
df.head()

In [ ]:
# Verifica: esegui senza modificare
assert list(df.columns) == ["Date", "Total Load [MW]"], "❌ df: devono restare solo Date e Total Load [MW]"
assert df["Date"].is_monotonic_increasing, "❌ df: ordina per Date"
assert df.loc[0, "Date"] == pd.Timestamp("2024-01-01 00:00"), "❌ df: la riga 0 deve essere il primo quarto d'ora del 2024; dopo sort_values serve reset_index(drop=True)"
print("✅ Step 2 completato")

<a id="sez-3"></a>
## 3. Ora legale: duplicati e buchi

Prima di aggregare controlliamo il passo tra un timestamp e il successivo. Due giorni all'anno qualcosa
non torna: l'ultima domenica di ottobre e l'ultima domenica di marzo, quando cambia l'ora.

<a id="es-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 3 · Pulizia dei duplicati dell'ora legale**

In questo step lasciamo una riga per timestamp e contiamo i quarti d'ora che mancano.

**Cosa fare.**

1. In `passi` metti la differenza tra ogni timestamp e il precedente (`diff()`) e conta quante volte
   compare ogni valore con `value_counts()`. Qual è il passo normale? Cosa sono gli altri due valori?
2. In `duplicati` metti le righe con un timestamp ripetuto, tutte le copie
   (`duplicated(keep=False)`). Che giorni sono? Che ore?
3. Ordina per `Date`, togli i duplicati con `drop_duplicates(subset="Date")` e reimposta l'indice.
4. In `buchi` metti le righe che arrivano dopo un salto più lungo di 15 minuti. Che giorni sono?

*Suggerimento:* L'ultima domenica di ottobre le 2:00-2:45 esistono due volte; l'ultima domenica di marzo non esistono.

</div>

In [ ]:
# 1. il passo tra un timestamp e il precedente, e quante volte compare ogni valore
passi = df["Date"].diff()
display(passi.value_counts())

# 2. le righe con un timestamp ripetuto, tutte le copie
duplicati = df[df["Date"].duplicated(keep=...)]
display(duplicati)

# 3. una riga per timestamp, indice da 0
df = df.sort_values("Date").drop_duplicates(subset=...).reset_index(drop=True)

# 4. le righe che arrivano dopo un salto più lungo di 15 minuti
buchi = df[df["Date"].diff() > ...]
buchi

In [ ]:
# Verifica: esegui senza modificare
assert len(duplicati) == 16, "❌ duplicati: 16 righe, 8 timestamp ripetuti in due copie (keep=False le tiene tutte)"
assert len(df) == 70168 and df["Date"].is_unique, "❌ df: 70.168 righe, una per timestamp"
assert len(buchi) == 2, "❌ buchi: 2 righe, le 3:00 delle due ultime domeniche di marzo"
print("✅ Step 3 completato")

<a id="sez-4"></a>
## 4. Aggregazioni temporali e profili medi

Useremo `Date` come colonna temporale e `Total Load [MW]` come variabile di interesse.

<a id="es-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 4 · Aggregazioni temporali e profili medi**

In questo step analizziamo la serie quartoraria cambiando punto di vista sul tempo. Costruiamo una
serie giornaliera e osserviamo i pattern medi legati all'ora del giorno e al giorno della settimana.

**Cosa fare.**

**Serie giornaliera (somma)**

1. Parti dal DataFrame quartorario `df`.
2. Imposta la colonna `Date` come indice temporale in `df_temp`.
3. Aggrega per giorno usando `resample("D")` e calcola la **somma** del carico.
4. Riporta `Date` come colonna e salva il risultato in `df_daily`. Aggiungi la colonna
   `Energia [MWh]`: la somma divisa per 4, perché un quarto d'ora a P MW vale P/4 MWh.
5. Visualizza la serie giornaliera con Plotly.

**Profilo medio giornaliero (media per ora del giorno)**

6. Torna alla serie quartoraria `df`: estrai l'**ora del giorno** dalla colonna `Date` e salvala in
   una nuova colonna `hour`.
7. Raggruppa per `hour` e calcola la **media** del carico in `profilo_orario`.
8. Visualizza il profilo medio giornaliero con Plotly.

**Profilo medio settimanale (media per giorno della settimana)**

9. Sempre partendo da `df`, estrai il **giorno della settimana** dalla colonna `Date` e salvalo in
   una nuova colonna `weekday`, con la convenzione `0 = lunedì, …, 6 = domenica`.
10. Raggruppa per `weekday` e calcola la **media** del carico in `profilo_settimanale`.
11. Visualizza il profilo medio settimanale con Plotly.

Guarda i tre grafici: in che periodi l'energia giornaliera scende? A che ore il carico è più alto,
e di quanto cala nel weekend?

*Suggerimento:* `dt.hour` e `dt.dayofweek` estraggono ora e giorno della settimana; `reset_index()` dopo `groupby` riporta la chiave come colonna.

</div>

In [ ]:
# serie giornaliera (punti 1-5)
df_temp = df.set_index("Date")
df_daily = df_temp["Total Load [MW]"].resample(...).sum()
df_daily = df_daily.reset_index()
df_daily["Energia [MWh]"] = ...
fig = px.line(df_daily, x="Date", y=..., title="Energia giornaliera, zona Nord")
fig.show()

# profilo medio giornaliero (punti 6-8)
df["hour"] = df["Date"].dt.hour
profilo_orario = df.groupby(...)["Total Load [MW]"].mean().reset_index()
fig = px.line(profilo_orario, x=..., y=..., markers=True, title="Profilo medio giornaliero del carico")
fig.update_xaxes(dtick=1)
fig.show()

# profilo medio settimanale (punti 9-11)
df["weekday"] = ...
profilo_settimanale = ...
profilo_settimanale["giorno"] = ["lunedì", "martedì", "mercoledì", "giovedì", "venerdì", "sabato", "domenica"]
fig = px.line(profilo_settimanale, x="giorno", y="Total Load [MW]", markers=True,
              title="Profilo medio settimanale del carico")
fig.show()

In [ ]:
# Verifica: esegui senza modificare
assert len(df_daily) == 731 and df_daily["Energia [MWh]"].between(250_000, 700_000).all(), "❌ df_daily: 731 giorni, con l'energia tra 250.000 e 700.000 MWh (somma dei quartorari divisa per 4)"
assert profilo_orario.shape == (24, 2), "❌ profilo_orario: 24 righe (le ore) e 2 colonne, hour e Total Load [MW]"
assert len(profilo_settimanale) == 7 and profilo_settimanale["Total Load [MW]"].idxmin() == 6, "❌ profilo_settimanale: 7 righe, con la domenica (6) come giorno più leggero"
print("✅ Step 4 completato")

<a id="sez-5"></a>
## 5. Recupero e integrazione della temperatura

Aggiungiamo la **temperatura a 2 metri** ricostruita per Milano. È una semplificazione: usiamo Milano
come punto "medio" per il Nord Italia. L'API
[Historical Weather](https://open-meteo.com/en/docs/historical-weather-api) di Open-Meteo non chiede
alcuna chiave e restituisce i dati orari su un intervallo di date.

La funzione qui sotto è già scritta: chiede i dati all'API con `requests.get` e restituisce la parte
`hourly` della risposta. Se la rete non risponde, il `try/except` legge
`../Dati/fallback/meteo_milano_2024_2025.json`: temperature di esempio, non misurate.

In [ ]:
import json
from pathlib import Path

import requests

In [ ]:
def scarica_temperatura(start, end):
    """Temperatura oraria a Milano tra due date (Open-Meteo): dizionario con le liste time e temperature_2m."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": 45.4642, "longitude": 9.19, "start_date": start, "end_date": end,
              "hourly": "temperature_2m", "timezone": "Europe/Rome"}
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        return response.json()["hourly"]
    except requests.RequestException:
        file_meteo = Path("../Dati/fallback/meteo_milano_2024_2025.json")
        print(f"API non raggiungibile: uso i dati di esempio di {file_meteo}")
        with open(file_meteo, encoding="utf-8") as f:
            return json.load(f)["hourly"]

<a id="es-5"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 5 · Recupero e integrazione della temperatura**

Ora dobbiamo integrare il dato di temperatura con il dato di carico. Per prima cosa va risolta la
differente granularità: la temperatura è oraria, il carico quartorario (upsampling da 1 ora a 15 minuti).

**Cosa fare.**

1. Ricava `start_date` e `end_date` dal carico: primo e ultimo giorno, come testo `AAAA-MM-GG`
   (`.min()`, `.max()` e `.strftime("%Y-%m-%d")`).
2. Chiama `scarica_temperatura(start_date, end_date)` e trasforma il risultato in `meteo` con
   `pd.DataFrame`.
3. Converti `time` in datetime, rinomina le colonne in `Date` e `Temperature_C`, imposta `Date` come
   indice e ordinalo con `sort_index()`.
4. Fai il resample a quartorario per poter integrare con il carico (`resample("15min")` +
   `interpolate(method="linear")`), riporta `Date` come colonna e salva il risultato in `meteo_15`.
5. Fai il merge con il DataFrame del carico `df`, tenendo tutte le sue righe, e salva il risultato in
   `df_full`.
6. Mancano i dati dell'ultima ora del 2025: riempi la temperatura con `ffill()`.
7. Visualizza la temperatura con `px.line()`.

*Suggerimento:* `how="left"` tiene tutte le righe del DataFrame di sinistra, la temperatura dove c'è.

</div>

In [ ]:
# 1. l'intervallo da chiedere, come testo AAAA-MM-GG
start_date = df["Date"].min().strftime("%Y-%m-%d")
end_date = ...
print("Intervallo:", start_date, "->", end_date)

# 2. la risposta dell'API: un dizionario con le liste time e temperature_2m
orario = scarica_temperatura(start_date, end_date)
meteo = pd.DataFrame(...)

# 3. date vere, nomi delle colonne, Date come indice
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.rename(columns={...})
meteo = meteo.set_index("Date").sort_index()

# 4. da orario a 15 minuti, con interpolazione lineare
meteo_15 = meteo.resample(...).interpolate(method="linear")
meteo_15 = meteo_15.reset_index()

# 5. merge con il carico, tenendo tutte le righe di df
df_full = pd.merge(df, meteo_15, on=..., how=...)

# 6. l'ultima ora del 2025 senza temperatura
df_full["Temperature_C"] = ...

# 7. la temperatura nel tempo
fig = px.line(df_full, x="Date", y="Temperature_C", title="Temperatura a Milano")
fig.show()

In [ ]:
# Verifica: esegui senza modificare
assert list(meteo_15.columns) == ["Date", "Temperature_C"], "❌ meteo_15: due colonne, Date e Temperature_C (dopo resample serve reset_index)"
assert len(df_full) == len(df), "❌ df_full: il merge con how=\"left\" tiene tutte le righe di df"
assert df_full["Temperature_C"].notna().all(), "❌ df_full: restano temperature mancanti, usa ffill()"
print("✅ Step 5 completato")

<a id="sez-6"></a>
## 6. Carico e temperatura

<a id="es-6"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 6 · Carico e temperatura**

Un punto per giorno: temperatura media sulle x, carico medio sulle y, un colore per mese. La forma che esce dice come il carico dipende dalla temperatura.

**Cosa fare.**

1. Porta `df_full` a medie giornaliere in `giornaliero`: `Date` come indice, le colonne
   `Total Load [MW]` e `Temperature_C`, `resample("D").mean()`, poi `reset_index()`.
2. Aggiungi la colonna `mese` con il nome del mese (`dt.month_name()`): così lo scatter ha un colore
   per mese invece di una scala continua.
3. Disegna lo scatter con `px.scatter`: temperatura sulle x, carico sulle y, `color="mese"`. Che
   forma ha? A che temperatura il carico è più basso?
4. Completa le cinque frasi della cella dopo la verifica, con i numeri e i grafici degli step.

*Suggerimento:* Con `dt.month`, che è un numero, Plotly colora con una scala continua; `dt.month_name()` dà un colore per mese.

</div>

In [ ]:
# 1. medie giornaliere di carico e temperatura
df_temp = df_full.set_index("Date")
giornaliero = df_temp[["Total Load [MW]", "Temperature_C"]].resample(...).mean()
giornaliero = giornaliero.reset_index()

# 2. il nome del mese, per colorare i punti
giornaliero["mese"] = ...

# 3. lo scatter: temperatura sulle x, carico sulle y
fig = px.scatter(giornaliero, x=..., y=..., color=..., hover_data=["Date"],
                 title="Carico medio giornaliero e temperatura media, zona Nord 2024-2025")
fig.show()

In [ ]:
# Verifica: esegui senza modificare
assert len(giornaliero) == 731, "❌ giornaliero: una riga per giorno, 731"
assert {"Total Load [MW]", "Temperature_C", "mese"} <= set(giornaliero.columns), "❌ giornaliero: servono le colonne Total Load [MW], Temperature_C e mese"
assert giornaliero["mese"].nunique() == 12, "❌ mese: i dodici mesi, da dt.month_name()"
print("✅ Step 6 completato")

**Il carico del Nord, 2024-2025**

1. I dati sono ...: ... righe pulite, ... doppioni di ottobre tolti, ... quarti d'ora di marzo mancanti.
2. L'energia giornaliera va da circa ... a circa ... MWh; i giorni più leggeri sono ...
3. Nella giornata media il carico è minimo alle ... e massimo alle ...
4. La domenica il carico medio sta intorno a ... MW, un giorno feriale intorno a ... MW.
5. Il carico è più basso intorno a ... °C e sale con il freddo e con il caldo; pesa di più ...

Fonte della temperatura: ... (Open-Meteo oppure i dati di esempio di `../Dati/fallback/`).

---

**Fine del notebook 11.**